# 🚀 06 · PPO 与 GRPO：策略梯度的工程形态（也是 LLM 对齐的底座）

> 面试定位：**PPO（Proximal Policy Optimization）** 是 RL 工程事实标准，更是 **RLHF/LLM 对齐** 的核心；
> **GRPO** 是 DeepSeek 在强化学习（R1）中采用的简化版。算法岗+LLM 岗双考。
> 本篇手写 clip 目标的数值实验，并把 PPO/GRPO 的 LLM 版公式讲透。
> 标记：🔴 必背。

---

## 1. PPO 要解决的问题（🔴）

### 朴素的策略梯度（REINFORCE）的问题

更新一步后策略变了 → 采样的轨迹**分布过时** → 用旧分布估计新策略梯度**有偏**。
幅度大时直接崩（**坍塌**：策略变坏、数据浪费、难以恢复）。

### TRPO 的解法（🟡 概念）

TRPO 用**约束**限制每步策略变化：

$$\max_\theta \mathbb{E}_{s,a\sim\pi_{\theta_{\text{old}}}} \left[\frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} A(s,a)\right] \quad \text{s.t.} \quad \mathbb{E}\left[\mathrm{KL}(\pi_{\theta_{\text{old}}} \| \pi_\theta)\right] \le \delta$$

- 每步只允许策略"挪动一点点" → **单调改进保证**（理论上）
- 缺点：KL 约束 + 共轭梯度求解复杂

### PPO 的工程化（🔴）

把"约束"换成"**裁剪（clip）惩罚**"，一行搞定：

$$L^{CLIP}(\theta) = \mathbb{E}\left[ \min\left( r_t(\theta) A_t,\; \operatorname{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\, A_t \right) \right]$$

其中**重要性权重** $r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$。

> **面试金句**：PPO = TRPO 的"约束换 clip"近似——当新旧策略偏差太大（$r_t$ 超出 $[1-\varepsilon, 1+\varepsilon]$）就把梯度**剪掉**，防止一步迈太远。

---


## 2. clip 目标的四种情形（🔴 手画理解）

| 情形 | A_t | r_t 位置 | 效果 |
|---|---|---|---|
| 好动作(A>0), 概率被提升 | >0 | r>1 | 保留提升, 但**限制在 1+ε 内** |
| 好动作(A>0), 概率被降低 | >0 | r<1 | 不惩罚(允许降低) |
| 坏动作(A<0), 概率被提升 | <0 | r>1 | 不阻止提升(给探索空间) |
| 坏动作(A<0), 概率被降低 | <0 | r<1 | 保留降低, 但**限制在 1-ε 内** |

一句话：**只裁剪"朝错误方向挪"的部分**——好动作别一下升太多，坏动作别一下降太多。ε 常见 0.1~0.3。

### PPO 的完整目标（三部分，背）

$$L(\theta) = L^{CLIP} + c_1\, L^{VF} - c_2\, S[\pi_\theta]$$

- $L^{VF}$：价值网络（Critic）的回归损失（MSE）
- $S$：**熵正则**（鼓励探索，防止策略过早确定）
- 熵项系数 $c_2$ 调小可提高探索性、调大更稳

### GAE 是 PPO 的标配（衔接 05 篇）

$$\hat{A}_t = \sum_{k=0}^{T} (\gamma\lambda)^k \delta_{t+k}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t), \quad \lambda \approx 0.95$$

---


In [ ]:
# ppo_clip_demo.py —— clip 目标数值演示: 对比 clip 前后梯度信号
import numpy as np

def ppo_clip_loss(r, A, eps=0.2):
    """r: 新旧策略概率比 πθ/πold  (向量);  A: 优势 (向量)"""
    surr1 = r * A
    surr2 = np.clip(r, 1 - eps, 1 + eps) * A
    return np.minimum(surr1, surr2)

# 场景: 好动作 A=1, 坏动作 A=-1, 分别看不同概率比 r 下的 loss
A_good, A_bad = 1.0, -1.0
rs = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 2.0])   # 概率比从降一半到翻倍

print(f"{'r(概率比)':<10}{'clip前(A=1)':>14}{'clip后(A=1)':>14}{'clip前(A=-1)':>15}{'clip后(A=-1)':>15}")
for r in rs:
    raw_g = r * A_good
    cl_g = ppo_clip_loss(np.array([r]), np.array([A_good]))[0]
    raw_b = r * A_bad
    cl_b = ppo_clip_loss(np.array([r]), np.array([A_bad]))[0]
    print(f'{r:<10.2f}{raw_g:>14.3f}{cl_g:>14.3f}{raw_b:>15.3f}{cl_b:>15.3f}')

print()
print('观察: r=2.0 时 clip 后(A>0)梯度被压在 1.2 封顶; (A<0)时 r 升高不惩罚')
print('→ 对称地, r=0.5 时 A<0 的下降被限制在 0.8 下限 —— 防止一步过头')


## 3. PPO 在 LLM（RLHF）中怎么用（🔴 LLM 岗必背）

### RLHF 三阶段（背）

```text
① 监督微调 SFT         : 用人工标注问答对训练基础模型
② 奖励建模 RM          : 用"人类偏好对"训练奖励模型 r_φ(x, y)（Bradley-Terry 排序损失）
③ PPO 优化             : 把语言模型当策略 π_θ, 生成 y ~ π_θ(·|x), 用 RM 打分做奖励
```

### LLM 版 PPO 目标（核心公式，背）

$$L(\theta) = \mathbb{E}_{(x,y)\sim\mathcal{D}}\Big[\underbrace{\min\big(r_t A_t,\; \operatorname{clip}(r_t, 1-\varepsilon, 1+\varepsilon) A_t\big)}_{\text{策略损失}} \; -\; \beta\,\underbrace{\mathrm{KL}\big(\pi_\theta \;\|\; \pi_{\text{SFT}}\big)}_{\text{与 SFT 模型不漂移}} \Big]$$

- 奖励 = 奖励模型分数 $r_\phi(x,y)$（可加长度/安全惩罚）
- **KL 项**（$\beta$ 系数）防止模型为了迎合奖励模型**说胡话/崩坏**（reward hacking）
- 优势 $A_t$ 由 Critic 对**整个句子**的 Q 值给出（token 级 GAE）

### 为什么 LLM 训练不用普通 RL 环境（背）

LLM 的"状态"是已生成的 token 序列，"动作"是下一个 token，环境是**词表 + 奖励模型**——每步奖励只在句末给出（稀疏），中间靠 GAE 把句末奖励**回传**给每一步的 token。

---


In [ ]:
# rlhf_kl_demo.py —— KL 正则的效果: 无 KL 会"刷爆"奖励模型漏洞(数值示意)
import numpy as np

# 模拟: 策略 π 在 4 个 token 上的分布 vs 参考(SFT)分布
pi_ref = np.array([0.4, 0.3, 0.2, 0.1])
np.random.seed(0)

# 无 KL 约束: 策略完全迎合奖励, 可能畸形(如死循环输出高奖励 token)
pi_greedy = np.array([0.99, 0.005, 0.004, 0.001])

def kl(p, q):
    return float(np.sum(p * np.log(p / q)))

def reward(pi):
    """奖励模型: 倾向 token0 (即模型发现 token0 给高分)"""
    return float(pi[0] * 10)

print(f'参考策略: 奖励 {reward(pi_ref):.2f}')
print(f'无 KL 约束策略: 奖励 {reward(pi_greedy):.2f}, 与参考 KL = {kl(pi_greedy, pi_ref):.3f}')
print('→ 分数高但分布极度偏离参考 → 输出退化成重复话术 (reward hacking)')
print()
# 带 KL 惩罚: 目标 = 奖励 - β·KL, 找平衡
betas = [0.0, 0.5, 1.0]
for b in betas:
    obj = reward(pi_greedy) - b * kl(pi_greedy, pi_ref)
    print(f'β={b}: 优化目标(奖励-βKL) = {obj:.3f}')
print('→ β 越大越守住参考分布; 实际训练中 β 是 KL 系数(0.01~0.1 量级, 且随训练调)')


## 4. GRPO：去 Critic 的 PPO 变体（🔴 DeepSeek-R1 用）

### 问题

PPO 需要 Critic（价值网络）——大模型上训练一个价值网络**昂贵且不稳定**（价值估计噪声大）。

### GRPO 的核心思想（背）

用**一组采样输出的经验均值**替代 Critic 估计基线：

$$\hat{A}_i = \frac{r_i - \operatorname{mean}(r_1, \dots, r_G)}{\operatorname{std}(r_1, \dots, r_G)}$$

对同一问题 $x$ 采样 $G$ 个回答 $y_1..y_G$，用这 $G$ 个奖励的**标准化得分**当优势：

- **去掉价值网络** → 省一半训练开销（R1 训练成本核心来源之一）
- 组内相对比较 = 奖励模型的"偏好"本质（排序而非绝对值）

### GRPO 目标（背）

$$L(\theta) = \mathbb{E}\Big[\frac{1}{G}\sum_{i=1}^{G} \min\big(r_i \hat{A}_i,\; \operatorname{clip}(r_i, 1-\varepsilon, 1+\varepsilon)\hat{A}_i\big)\Big] - \beta\, \mathrm{KL}(\pi_\theta \| \pi_{\text{ref}})$$

$r_i = \dfrac{\pi_\theta(y_i|x)}{\pi_{\text{ref}}(y_i|x)}$ 是策略与参考模型的概率比（**不是**与 old 策略比，GRPO 省了重要性采样里的 old 快照，参考模型固定）。

### PPO vs GRPO（🔴 面试对比表）

| 维度 | PPO(RLHF) | GRPO(DeepSeek R1) |
|---|---|---|
| 基线/优势 | Critic 价值网络 | **组内奖励标准化**（无 Critic） |
| 训练开销 | 高（价值网络大） | 低（省一路） |
| 方差控制 | GAE 多步 | 组内 G 越大越稳 |
| 适用 | 通用 RL | 奖励模型可批量的生成任务（LLM 对齐） |
| 出处 | OpenAI/RLHF | DeepSeekMath / R1 论文 |

---


In [ ]:
# grpo_advantage.py —— GRPO 组内标准化优势 vs PPO 的 Critic 优势: 数值对比
import numpy as np

np.random.seed(4)
# 同一问题采样 4 个回答, 奖励模型打分
rewards = np.array([7.0, 3.5, 9.0, 5.5])

# GRPO: 组内标准化
mean, std = rewards.mean(), rewards.std()
adv_grpo = (rewards - mean) / (std + 1e-6)

# 模拟 PPO: 用(理想)Critic 估计基线(这里假设 Critic 估计每个 x 的期望价值)
critic_baseline = np.array([6.0, 6.0, 6.0, 6.0])   # 理想 Critic 给同一基线
adv_ppo = rewards - critic_baseline

print('奖励模型打分:', rewards)
print('GRPO 优势(标准化):', np.round(adv_grpo, 3))
print('PPO 优势(Critic基线):', np.round(adv_ppo, 3))
print()
print('共同点: 都是"相对好坏"信号(正=比平均好, 负=比平均差)')
print('差异: GRPO 用组内均值/方差, 天然有尺度无关性; PPO 依赖 Critic 估计质量')


## 5. PPO 训练的经验坑（工程向背诵）

1. **clip ε 太小** → 学得慢；太大 → 接近 TRPO 难题（建议 0.2 起步）
2. **GAE λ 调小**（如 0.9）方差小但偏差大，生成类任务常用 0.95~0.99
3. **价值网络过拟合** → PPO 变慢；用 **Value Clipping**（价值损失也 clip）或减小 Critic 学习率
4. **奖励归一化**（均值 0 方差 1）→ 稳定；奖励模型分数分布不一致时尤其重要
5. **KL 惩罚是硬约束还是软惩罚**：PPO-kl 有 penalty/adaptive 两种；自适应版本（目标 KL 内调 β）更稳
6. **batch 与 mini-batch**：一版数据内多次 epoch 更新（3~10 epoch），每次 mini-batch 采样

### 面试追问速答

1. **Q: PPO 和 TRPO 区别？** A: TRPO 用硬 KL 约束+共轭梯度；PPO 用 clip 惩罚目标，简单高效。
2. **Q: clip 为什么能防崩塌？** A: 概率比超出 [1-ε,1+ε] 时梯度被截断 → 单步更新幅度有界。
3. **Q: LLM 的 KL 项放哪？** A: 目标函数里 -β·KL(πθ‖πref)，约束生成分布不偏离 SFT 太远。
4. **Q: GRPO 为什么去 Critic？** A: 大模型价值网络贵且噪声大；组内奖励标准化提供免训练基线。
5. **Q: 熵项作用？** A: 防止策略坍缩成确定性、鼓励探索；LLM 中对应保持生成多样性。

---

## 6. 自测清单

- [ ] L1：默写 PPO clip 目标与重要性权重公式
- [ ] L1：画 clip 目标四象限（好/坏动作 × 升/降概率）
- [ ] L1：说出 RLHF 三阶段与 LLM 版目标里的 KL 项作用
- [ ] L2：对比 PPO 与 GRPO 的优势计算（Critic vs 组内标准化）
- [ ] L2：解释 GAE 公式与 λ 的偏差-方差旋钮

>  下一篇 `07-连续控制DDPG与SAC` + `08-探索利用`：完整覆盖面试剩余高频。


## 7. PPO 的两种变体：clip 版与 KL 惩罚版（🟡 掌握差异）

### PPO1: KL 惩罚

$$L = \mathbb{E}\left[ \frac{\pi_\theta}{\pi_{\theta_{old}}} A_t \right] - \beta\, \mathrm{KL}(\pi_{\theta_{old}} \| \pi_\theta)$$

- 惩罚项限制步长；β 可**自适应**（目标 KL 与实际 KL 比较后调 β）

### PPO2: clip（主流）

$$L = \mathbb{E}\left[\min(r_t A_t,\; \operatorname{clip}(r_t, 1-\varepsilon, 1+\varepsilon) A_t)\right]$$

- 不用算 KL，只需概率比 → 常是默认选择

### 对比（背）

| | PPO-KL | PPO-clip |
|---|---|---|
| 约束方式 | 软惩罚 | 硬裁剪 |
| 实现 | 需算 KL | 只需 ratio |
| 调参 | β(+自适应) | ε |
| 现状 | 教学用 | **工程默认** |

---


In [ ]:
# kl_adaptive.py —— 自适应 KL 惩罚: β 随实际 KL 自动升降
import numpy as np

def adaptive_kl_step(target_kl=0.01, beta=1.0, kl_now=0.0):
    """按 PPO 论文: KL 超目标 → 加大 β(惩罚); 低于目标 1.5 倍 → 减小 β"""
    if kl_now > target_kl * 1.5:
        beta *= 2.0
    elif kl_now < target_kl / 1.5:
        beta /= 2.0
    return beta

beta = 1.0
print('模拟 KL 波动 → β 自适应调整:')
print(f'{"时刻":<6}{"实际 KL":>10}{"β":>10}')
for t, kl in enumerate([0.005, 0.02, 0.03, 0.012, 0.008, 0.004, 0.002]):
    beta = adaptive_kl_step(0.01, beta, kl)
    print(f'{t:<6}{kl:>10.4f}{beta:>10.3f}')
print()
print('→ KL 过大加倍惩罚(防爆炸), 过小减半(不拖慢) —— PPO 论文的自动调参')


## 8. PPO 完整最小实现：clip 目标在小环境上跑（🔴 动手实现）

### 结构

```text
Actor(softmax) --- ratio = πθ(a|s)/πold(a|s)
Critic(V 表/线形) --- 优势 = GAE
目标: min(r·A, clip(r,1±ε)·A) - c2·H
每 epoch: 旧策略快照 → 采样一批 → 多 epoch 更新
```

下面实现一个"PPO-表格+线性"版本在 4×4 网格上训练，见证 clip 让训练稳定。

---


In [ ]:
# ppo_mini.py —— 迷你 PPO: clip 目标 + GAE, 4x4 网格
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def step(s, a):
    if s == N - 1:
        return N - 1, 0.0
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return s, -1.0
    s2 = nr * SIZE + nc
    return (N - 1, 10.0) if s2 == N - 1 else (s2, -1.0)

def feats(s):
    x = np.zeros(N); x[s] = 1.0
    return x

def act_prob(W, s):
    x = feats(s)
    logits = W @ x
    logits -= logits.max()
    e = np.exp(logits)
    return e / e.sum(), x

def collect_rollout(W, max_steps=1000):
    """一版数据: 记 (s,a,r) 直到若干条轨迹"""
    traj = []
    total = 0.0
    for ep in range(10):
        s = 0
        while s != N - 1:
            p, _ = act_prob(W, s)
            a = int(np.random.choice(4, p=p))
            s2, r = step(s, a)
            traj.append((s, a, r))
            s = s2
            total += r
            if s == N - 1:
                break
    return traj, total / 10.0

def compute_returns(rewards):
    """简单 MC 回报(无价值函数版) 便于演示 clip"""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + GAMMA * G
        out.append(G)
    return np.array(out[::-1])

def ppo_update(W, data, epsilon=0.2, alpha=0.05, epochs=3):
    """data: (s, a, G)[]; 多 epoch 小步更新"""
    for _ in range(epochs):
        idx = np.random.permutation(len(data))
        for i in idx:
            s, a, G = data[i]
            p_old, _ = act_prob(W, s)
            # 旧策略概率(更新前快照近似: 用当前 W 即"旧"——迭代内 W 变化小, 演示足够)
            for _step in range(1):
                p_new, _ = act_prob(W, s)
                ratio = p_new[a] / (p_old[a] + 1e-8)
                surr1 = ratio * G
                surr2 = np.clip(ratio, 1 - epsilon, 1 + epsilon) * G
                loss = -min(surr1, surr2)
                # 手动梯度: d(loss)/d(logit_a) ≈ -(1 or 0)·G·p_a 的近似, 走简单数值上升
                W[a, :] += alpha * (min(surr1, surr2)) * feats(s)   # 简化: 与 REINFORCE 同形
            return W  # 每 epoch 只演示一轮(教学简化)

np.random.seed(5)
W = np.random.RandomState(2).randn(4, N) * 0.1
for pg in range(30):
    data, avg = collect_rollout(W)
    W = ppo_update(W, data)
    if pg % 5 == 0:
        print(f'PPO 第 {pg:2d} 轮: 平均轨迹奖励 {avg:6.2f}')

print()
print('迷你 PPO 让 clip 限幅; 完整版还需 GAE + Critic + 多 epoch, 见 04-LLM/21 篇')


## 9. LLM-RLHF 的奖励模型细节（🔴 必背）

### Bradley-Terry 排序损失（背）

奖励模型 $r_\phi$ 训练数据是偏好对 $(x, y_w, y_l)$：

$$\mathcal{L}(\phi) = -\mathbb{E}\left[\log \sigma\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right)\right]$$

- 学习"好回答比坏回答分数高"
- 与交叉熵同构（二分类）

### RM 的常见问题（背三个）

1. **偏好标注不一致** → 用多标注投票/KTO
2. **奖励模型被 hack**：策略学到"高分话术"而非"好答案" → KL 正则 + 多样本评估
3. **奖励分布漂移**：训练中奖励均值持续上升 → 归一化/调 β

---


In [ ]:
# rm_loss.py —— Bradley-Terry 奖励模型损失: 手写 + 直观
import numpy as np

def bt_loss(r_w, r_l):
    """log σ(r_w - r_l), Mean 版本"""
    return -np.log(1 + np.exp(-(r_w - r_l)))

# 造打分: 好回答 8 分, 坏回答 2 分
r_w, r_l = 8.0, 2.0
print(f'偏好对 (w=8, l=2): logσ(6) = {bt_loss(r_w, r_l):.4f} (损失小=好模型)')
print(f'模型打分反了 (w=2, l=8): logσ(-6) = {bt_loss(2.0, 8.0):.4f} (损失大)')
print()
# 梯度: 提升好回答分数、降低坏回答分数
for r in np.linspace(0, 10, 11):
    print(f'r_w={r:4.1f} r_l=2: loss={bt_loss(r, 2.0):6.3f}')
print()
print('→ RM 训练本质: 拉开 y_w 与 y_l 的分数差; PPO 阶段用这个分数当奖励')


## 10. 奖励黑客与 KL 防漂移的数值实验（🔴 深化）

### 复习

PPO 目标里 $-\beta\,\mathrm{KL}(\pi_\theta \| \pi_{\text{ref}})$ 惩罚"离 SFT 太远"。无 KL 时策略会**钻奖励模型的空子**（如答非所问刷高分）。

### 实验设计

模拟两种策略分布：①正常（贴近参考）②刷分（单 token 概率 0.99）——比较"奖励"与"总目标（奖励-βKL）"。

---


In [ ]:
# kl_tradeoff.py —— 奖励 vs KL 的目标权衡图(数值)
import numpy as np

def kl(p, q):
    return float(np.sum(p * np.log((p + 1e-9) / (q + 1e-9))))

pi_ref = np.array([0.4, 0.3, 0.2, 0.1])
# 策略 A: 温和提升 token0; 策略 B: 极端刷分
pi_A = np.array([0.55, 0.22, 0.15, 0.08])
pi_B = np.array([0.99, 0.004, 0.004, 0.002])

def reward(pi, alpha=0.9):
    """奖励模型倾向 token0(第1位), 真实性等高奖励"""
    return float(alpha * pi[0] + (1 - alpha) * sum(pi * np.array([1.0, 0.6, 0.3, 0.1])))

print(f'{"策略":<10}{"奖励":>8}{"KL(π‖ref)":>12}{"目标(β=0.5)":>14}')
for name, pi in [('参考', pi_ref), ('温和A', pi_A), ('刷分B', pi_B)]:
    r = reward(pi)
    k = kl(pi, pi_ref)
    print(f'{name:<10}{r:>8.3f}{k:>12.3f}{r - 0.5 * k:>14.3f}')
print()
print('观察: 刷分 B 奖励最高, 但扣 KL 后总目标变差(A 最优)')
print('→ KL 正则把"刷分"变成"不划算", 这就是防 reward hacking 的机制')


## 11. GRPO 的组大小 G 与方差（🟡 实验）

### 直觉

组内标准化优势 $\hat{A}_i = (r_i - \bar{r})/\mathrm{std}$：
- G 大 → 均值/标准差更稳 → 优势估计方差更小
- G 小（如 2）→ 噪声大（尤其 std 接近 0 时）

### 实验

固定奖励分布，扫 G ∈ {2, 4, 8, 16}，统计标准化优势的方差。

---


In [ ]:
# grpo_group_size.py —— GRPO 组大小 G 对优势方差的影响
import numpy as np

np.random.seed(0)
# 奖励模型给同一问题的打分分布(模拟): 均分 5, 标准差 2 的高斯
mu, sd = 5.0, 2.0

print(f'{"G":<6}{"优势均值":>10}{"优势标准差":>12}')
for G in [2, 4, 8, 16, 32]:
    advs = []
    for _ in range(2000):
        rewards = np.random.normal(mu, sd, G)
        adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
        advs.extend(adv)
    advs = np.array(advs)
    print(f'{G:<6}{advs.mean():>10.3f}{advs.std():>12.3f}')

print()
print('解读: 优势均值始终≈0(组内标准化); G 越大单组估计越稳(优势 std→1)')
print('→ 实践中 G=4~8 是 DeepSeek 系常用选择(显存与方差折中)')


## 12. DPO：不需要强化学习的对齐（🟡 对比了解）

### 核心洞察

把 RLHF 的奖励最大化目标**解析解**代回去 → 目标变成只依赖偏好对的**分类损失**：

$$\mathcal{L}_{DPO} = -\log\sigma\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

### DPO vs PPO（背）

| | PPO-RLHF | DPO |
|---|---|---|
| 训练 | 需 RM + 在线采样 + Critic | 只需偏好对（离线） |
| 稳定性 | 需调很多超参 | 简单稳定 |
| 上线成本 | 高 | 低 |
| 性能 | 通常更强（在线探索） | 足够好（离线弱一些） |
| 现状 | 大厂主力 | 开源主力 |

> **面试金句**："DPO 把 RL 目标闭式求解进了损失函数——省掉 RM 与在线采样，代价是失去在线探索的收益；RLHF 仍是质量上限最高的路线。"

---


## 13. 面试追问速答扩展（🔴）

1. **Q: PPO 的 clip 与 TRPO 约束等价吗？** A: 一阶等价近似；clip 是代理目标而非严格约束。
2. **Q: 为什么要存旧策略快照？** A: 概率比需要固定分母 πold——更新过程中分母不变。
3. **Q: GAE λ 对 LLM 任务影响？** A: λ 大更接近 MC（长程信用分配好）；λ 小更稳。
4. **Q: RLHF 的 KL 是 KL(πθ‖πsft) 还是反过来？** A: 论文用 KL(新‖旧/参考)，方向影响惩罚强度。
5. **Q: GRPO 的 G 为什么不能无限大？** A: 显存/算力随 G 线性涨；且同一问题重复采样次数有限。
6. **Q: 为什么 GRPO 省显存？** A: 省掉价值网络（与策略同规模的 critic）——大模型上加价值网络显存翻倍。
7. **Q: RM 与 KL 的平衡怎么调？** A: β 大 → 稳但进步慢；β 小 → 进步快但易 hack；监控 KL 曲线。
8. **Q: PPO 里熵项对 LLM 的作用？** A: 保持生成多样性、防止坍缩成固定话术。
9. **Q: 参考模型多大？** A: 常与策略同规模（冻结）→ 显存翻倍；可用 LoRA 近似省显存。
10. **Q: 离线 vs 在线对齐？** A: 在线（RLHF/GRPO）能拿到新数据，质量上限高；离线（DPO）简单稳定。

---

## 14. 自测清单（06 篇收官）

- [ ] L1：默写 PPO clip 目标、GRPO 优势公式、BT 损失
- [ ] L1：讲清 KL 防 reward hacking 的机制
- [ ] L1：说清 DPO 与 RLHF 的取舍
- [ ] L2：手写 GAE 递归实现并在轨迹上验证
- [ ] L2：GRPO 组大小扫描结论能口头复述
- [ ] L3：把 PPO/GRPO 与 04-LLM 的 21 篇推导串起来


## 15. PPO/GRPO 公式总表（🔴 考前速背）

| 概念 | 公式 |
|---|---|
| 重要性权重 | $r_t(\theta)=\pi_\theta(a_t\mid s_t)/\pi_{\theta_{old}}(a_t\mid s_t)$ |
| PPO-clip | $\min(r_t A_t,\; \mathrm{clip}(r_t,1-\varepsilon,1+\varepsilon)A_t)$ |
| PPO 总目标 | $L^{CLIP}+c_1 L^{VF}-c_2 S[\pi_\theta]$ |
| RLHF 奖励 | $r_\phi(x,y)-\beta\,\mathrm{KL}(\pi_\theta\|\pi_{\mathrm{SFT}})$ |
| BT 损失 | $-\log\sigma(r_w-r_l)$ |
| GRPO 优势 | $\hat A_i=(r_i-\bar r)/\mathrm{std}(r_1..r_G)$ |
| GRPO 目标 | $\mathbb{E}[\frac{1}{G}\sum_i \min(r_i\hat A_i,\mathrm{clip}\cdot\hat A_i)]-\beta\mathrm{KL}(\pi_\theta\|\pi_{\mathrm{ref}})$ |


## 16. RLHF vs DPO vs GRPO 一页对比（🔴 面试终板）

| | RLHF(PPO) | DPO | GRPO(R1) |
|---|---|---|---|
| 奖励 | RM 打分 | 隐式（偏好对） | RM 打分 |
| 基线 | Critic(GAE) | 无 | 组内标准化 |
| 在线采样 | 需要 | 不需要 | 需要 |
| 显存 | 高(+RM+Critic) | 低 | 中(去Critic) |
| 稳定性 | 难调 | 简单 | 中等 |
| 适用 | 高质量上限 | 资源受限 | 可验证任务(推理) |

> **记忆锚点**：DPO 离线稳但无探索；RLHF 在线强但贵；GRPO 用组内比较替代 Critic，是 RLHF 的"省显存版"。


## 17. 易错点清单（🔴 PPO/GRPO 篇）

1. **ratio 的分母是"旧策略"**：更新中固定 π_old（快照），不能每步重算
2. **GRPO 的 ratio 分母是参考模型 π_ref**：不是 old 策略（GRPO 无 old 快照）
3. **KL 方向**：KL(πθ‖πref) 是"新对参考"；DPO 里 log 里的 β 系数别丢
4. **clip 是惩罚不是约束**：TRPO 硬约束；PPO 软裁剪——面试直接问区别
5. **done 处理**：轨迹终点 TD 目标无后继价值（γV(s')=0）
6. **GAE 从尾部递归**：A_t=δ_t+γλA_{t+1}，方向别写反


## 18. 考前 3 分钟速背（🔴 PPO/GRPO 篇）

```text
PPO = TRPO 的 clip 化: min(r·A, clip(r,1±ε)·A)
三件套: clip + GAE + 熵正则
RLHF: SFT → RM(BT) → PPO(+βKL)
KL 防 hack; G=4~8 组内标准化 -> GRPO(去Critic)
DPO: 闭式解偏好对, 离线简单, 无在线探索
R1 风格: 可验证任务 + GRPO + 长 CoT 奖励
```


## 19. 常见追问组合演练（🔴 面试对练）

- **A**: "PPO 和 TRPO 你选哪个？" → PPO（简单、表现接近、少调参）
- **B**: "KL 惩罚和 clip 能一起用吗？" → 可以（有些实现两者都加，更稳）
- **C**: "LLM 上你更看好 RLHF 还是 DPO？" → 质量优先 RLHF；资源受限 DPO；推理增强 GRPO
- **D**: "奖励模型怎么防被 hack？" → βKL + 评估集盲评 + 长度惩罚 + 多样本


In [ ]:
# llama_translate.py —— PPO/GRPO 超参速查(print-only 备忘)
config = {
    'PPO-clip': {'ε': 0.2, 'γ': 0.99, 'GAE λ': 0.95, 'lr': '3e-4'},
    'PPO-KL': {'β': '自适应(目标KL 0.01±)', 'target': 0.01},
    'RLHF': {'RM 初始': 'SFT 权重', 'βKL': '0.01~0.1', '采样': 'per-prompt batch'},
    'GRPO': {'G': '4~8', '优势': '(r-mean)/std', '去 Critic': True},
    'DPO': {'β': '0.1~0.5', '数据': '偏好对离线', '在线': False},
}
print('LLM 对齐超参速查表:')
for k, v in config.items():
    print(f'  {k:<10}: {v}')
print()
print('记忆锚点: PPO 想 ε 和 λ; GRPO 想 G 和去Critic; DPO 想 β 和离线')


## 20. 项目脑暴：给 MiniMind 加一个 GRPO 微调（🟡 高阶实践）

```text
数据: 数学题 x 1000 (答案可验证)
策略: MiniMind(或小模型) 生成 CoT
奖励: 答案正确 + 长度惩罚; 组内 G=4
优化: GRPO(clip + 组内优势) 或 DPO(离线)
评估: 正确率 vs KL 漂移 vs 生成多样性
产出: 正确率提升曲线 + 采样案例对比
```

> 这是"LLM 岗+RL 岗"双杀的简历项目——04-LLM/21 篇有公式推导，这里给出工程路线。


## 21. 易错自查（🔴 上机前）

- [ ] ratio 分母是旧策略（快照），不是参考模型（GRPO 例外）
- [ ] done 处理：终点无 γV(s')
- [ ] KL 方向：KL(πθ‖πref)
- [ ] GRPO 优势有 std 分母（别只减均值）
- [ ] 熵项系数别漏（很多 PPO 实现默认带）


## 22. LLM 对齐开源生态速览（🟢 找工作有用）

| 框架 | 特点 | 适用 |
|---|---|---|
| TRL (HF) | PPO/DPO/GRPO 都有 | 快速原型 |
| LLaMA-Factory | SFT/RM/DPO/PPO 全家桶 | 科研/实验 |
| OpenRLHF | 大规模 RLHF 分布式 | 生产 |
| verl (字节) | RLHF 框架，GRPO 支持好 | 推理 RL |
| veRL/DeepSpeed-Chat | 大模型 RLHF | 大集群 |

> 简历写"用过 TRL/LLaMA-Factory 跑过 DPO/GRPO"，面试官默认你会 LLM×RL 落地。


In [ ]:
# grpo_pseudo.py —— GRPO 训练循环伪代码(背诵版, 打印对照)
pseudo = '''
for step in range(T):
    prompts = sample_batch()                    # x1..xB
    for i in range(B):
        y_i1..y_iG = policy.sample(prompts[i], G=4)   # 每组 G 个回答
        r_ig = reward_model(prompts[i], y_ig)          # RM 打分
        r_ig = normalize_length(r_ig)                 # 长度惩罚
    for i in range(B):
        A_ig = (r_ig - mean(r_i)) / (std(r_i) + eps)  # 组内标准化优势
        ratio = policy(y_ig|x) / ref(y_ig|x)          # 与参考模型比
        loss = -mean(min(ratio*A, clip(ratio,1±eps)*A)) + β·KL
    policy.update(loss)
'''
print(pseudo)
print('默写要点: G 组采样 → RM → 组内优势 → ratio对照ref → clip+KL')


## 23. 面试实战演练：R1 为什么选择 GRPO（🔴 60 秒版）

> "DeepSeek-R1 的目标是推理增强：输入数学/代码题，期望模型自己写出长 CoT 并给出正确答案。
> 这类任务奖励可验证（答案对错），可以批量采样多条回答；GRPO 用同一问题的 G 条回答做组内比较，
> 优势 = 标准化分数，替代了 PPO 里昂贵且不稳的价值网络——训练成本降一半且效果更好。
> 同时 KL 约束保证不偏离参考模型，避免奖励黑客。"

**要点**：任务可验证 → 批量采样 → 组内比较 → 去 Critic。四步全中。


## 24. 章末寄语（🟢）

PPO 是 RL 的"瑞士军刀"，GRPO 是 LLM 对齐的"省电版"。至此你已覆盖 RL 算法岗 + LLM 对齐岗的核心弹药。


## 25. 全篇公式自查（🔴 默写后对照）

- [ ] PPO clip 目标一行
- [ ] GAE 递归一行
- [ ] BT 损失一行
- [ ] GRPO 优势一行
- [ ] RLHF 总目标（含 βKL）一行
- [ ] DPO 损失一行（知道即可）

> 六个"一行"，都是面试 30 秒题——全部能盲写才算过。


## 26. 面试对比三板斧（🔴 对齐篇）

1. **RLHF vs DPO**：在线+RM+贵 vs 离线+偏好对+简单
2. **PPO vs GRPO**：Critic+GAE vs 组内标准化+去Critic
3. **推理 RL vs 通用 RLHF**：可验证奖励 vs 人类偏好奖励

> 三组对比覆盖"LLM 对齐岗"所有高频追问。


In [ ]:
# align_compare.py —— 对齐三对比一览(打印)
print('RLHF vs DPO:')
print('  RLHF: SFT→RM(BT)→PPO(+βKL)；在线探索，质量上限高，贵')
print('  DPO: 偏好对闭式解；离线简单稳定，无探索，上限略低')
print()
print('PPO vs GRPO:')
print('  PPO: Critic 估基线 + GAE；通用但贵')
print('  GRPO: 组内奖励标准化；去 Critic，省显存（R1 用）')
print()
print('推理RL vs 通用RLHF:')
print('  推理RL: 答案可验证（对错），鼓励长 CoT（R1 风格）')
print('  通用RLHF: 人类偏好排序，覆盖开放生成')


## 27. 章末寄语（🟢）

PPO/GRPO 篇收官。核心资产：clip 一行 + KL 防 hack + GRPO 组内优势 + 对齐三对比。


## 28. 全模块主线回顾（🔴 串联 00-06）

```text
00 定义世界(MDP) → 01 有模型求解(DP) → 02 无模型预测(MC/TD)
→ 03 无模型控制(Q/SARSA) → 04 深度价值(DQN) → 05 深度策略(PG/AC)
→ 06 工程标准+LLM 对齐(PPO/GRPO) → 07 面试实战(90问+手写)
```

> 面试官问"你的 RL 知识框架"——把这条线 30 秒讲完，等于把目录背给他听。


## 29. 数字敏感度速查（🔴 PPO/GRPO 篇终版）

| 参数 | 典型值 | 说明 |
|---|---|---|
| clip ε | 0.2 | 0.1~0.3 |
| GAE λ | 0.95 | 0.9~0.99 |
| KL β(RLHF) | 0.01~0.1 | 可自适应 |
| KL β(DPO) | 0.1~0.5 | 常需扫描 |
| GRPO G | 4~8 | 显存与方差折中 |
| 熵系数 | 0.01 | 保多样性 |
| rollout batch | 2048~4096 | 一版数据 |


## 30. 章末终版寄语（🟢）

> PPO 是 RL 的"瑞士军刀"、GRPO 是 LLM 对齐的"省电版"、DPO 是"轻量版"——三件套加身，无论面试官从哪个角度问对齐，你都有公式、对比与工程细节可讲。


## 31. 模块衔接（🔴）

06 篇把所有知识收束到 LLM 对齐；07 篇是 90 问 + 手写 + 开放题的总决战。至此 8 篇闭环。


## 32. LLM 对齐变种谱系（🔴 PPO / KL / GRPO / RLOO / DPO 推导关系）

### 谱系树

```text
RLHF(PPO+RM) ─→ PPO-clip(默认) ─→ RLOO(留一基线)
     │                              └→ GRPO(组内标准化, 去Critic)
     └→ DPO(闭式解, 离线)          └→ KTO(无需配对偏好)
```

### 一条主线推导（背）

- PPO 需要 Critic：$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$（价值网络市值）
- RLOO 去掉 Critic：基线 = 组内"排除自己"的均值 $\bar r_{-i}$
- GRPO 再去掉"留一"，用全组：$\hat A_i = \frac{r_i - \bar r}{\mathrm{std}(r_1..r_G)}$
- 每一步都在回答同一问题：**"用更便宜的方式逼近优势 $A_t$"**——从学价值 → 组内统计。


## 33. TRPO → PPO：代理目标与 clip 的推导（🔴 面试手推主菜）

### 背景

TRPO 硬约束：$\max_\theta \mathbb{E}[r_t(\theta)\hat A_t] \quad\mathrm{s.t.}\quad \mathrm{KL}(\theta_{old}\|\theta)\le\delta$

### 三步推导（背）

```text
① 重要性采样改写目标(用旧策略采样):
   L^CPI(θ) = E_t[ ( πθ(a|s)/πold(a|s) ) · Â_t ] = E_t[ r_t(θ)·Â_t ]
   ↑ 唯一开销: 每步重算概率比 r_t(θ) —— 不需要重采数据!

② TRPO 的 KL 约束在 PPO 里换成两种近似:
   PPO1:  罚项   L = L^CPI - β·KL(θold‖θ)      (β 可自适应)
   PPO2:  clip 下界  L^CLIP = E[ min(r_t Â_t, clip(r_t,1±ε)Â_t) ]

③ clip 的直觉(为什么对): r_t 超出 [1-ε, 1+ε] 时目标被"压平",
   梯度不再把 θ 推向"一步迈太大"的方向 → 步长自动受限 ∎
```

> **面试金句**："PPO 用'概率比 + 裁剪'复刻了 TRPO 的信任域，代价只是每个样本多算一次新旧概率比——这是它成为工程默认的全部理由。"


In [ ]:
# fig_ppo_clip.py —— 过程图①: PPO clip 目标曲面(ratio vs 优势方向)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

r = np.linspace(0.0, 2.0, 200)
eps = 0.2

def l_clip(r, A, eps=0.2):
    return np.minimum(r * A, np.clip(r, 1 - eps, 1 + eps) * A)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8), dpi=150)
for ax, A in zip(axes, [1.0, -1.0]):
    ax.plot(r, r * A, ls='--', color='gray', label='无 clip（线性）')
    ax.plot(r, l_clip(r, A, eps), lw=2.2, color='crimson', label=f'clip(1±{eps})')
    ax.axvspan(1 - eps, 1 + eps, color='gold', alpha=0.25, label='信任区域')
    ax.axvline(1.0, color='black', lw=1, ls=':')
    ax.set_title(f'优势 A={A:+.0f}', fontsize=11)
    ax.set_xlabel('概率比 r_t(θ)'); ax.set_ylabel('代理目标')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle('PPO clip: 信任区域外目标被压平 → 梯度受限', y=1.02)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'ppo_clip_curve.png'), dpi=150)
print('已保存 images/ppo_clip_curve.png')
plt.close(fig)
plt.show()


In [ ]:
# fig_kl_beta.py —— 过程图②: KL 系数 β 的权衡(奖励 vs 偏移)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 模拟: β 越大 → KL 偏移越小, 但奖励增益也小
betas = np.array([0.0, 0.01, 0.05, 0.1, 0.2, 0.5])
kl = np.array([0.42, 0.33, 0.22, 0.15, 0.09, 0.04])        # KL(θ‖ref) 示意
reward_gain = np.array([6.0, 5.2, 4.1, 3.0, 1.8, 0.6])     # RM 得分增量示意

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=150)
ax2 = ax.twinx()
ax.plot(betas, kl, 'o-', color='royalblue', label='KL(θ‖ref)（左轴）')
ax2.plot(betas, reward_gain, 's-', color='darkorange', label='奖励增益（右轴）')
ax.set_xlabel('KL 系数 β'); ax.set_ylabel('KL 偏移', color='royalblue')
ax2.set_ylabel('奖励模型得分增量', color='darkorange')
ax.set_title('β 权衡: 惩罚太松→漂移大(RM被hack); 太紧→不进步')
ax.legend(loc='upper right'); ax2.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'rlhf_kl_beta.png'), dpi=150)
print('已保存 images/rlhf_kl_beta.png')
plt.close(fig)
plt.show()


## 📸 本章推导配图（教学/images/）

![PPO clip 目标曲面: 信任区域外目标被压平](images/ppo_clip_curve.png)

> PPO clip 目标曲面: 信任区域外目标被压平

![KL 系数 β 权衡: 漂移 vs 奖励增益](images/rlhf_kl_beta.png)

> KL 系数 β 权衡: 漂移 vs 奖励增益
